In [1]:
import mysql.connector
import pandas as pd
from mysql.connector import Error

In [2]:
try:
    conn = mysql.connector.connect(
        port="3307",
        user="root",
        password="", 
        database="uber_eats_bangalore"
    )
    cursor = conn.cursor()
    print("Connected to MySQL!")
except Error as e:
    print("Connection error:", e)

Connected to MySQL!


In [3]:
def run_query(query, cols=None):
    cursor.execute(query)
    result = cursor.fetchall()
    df = pd.DataFrame(result, columns=cols)
    return df

Top Rated Locations

In [4]:
q1 = """
SELECT location, ROUND(AVG(rate), 2) AS avg_rating, COUNT(*) AS num_restaurants
FROM restaurants 
GROUP BY location 
HAVING COUNT(*) >= 5
ORDER BY avg_rating DESC 
LIMIT 10;
"""
display(run_query(q1, ['Location', 'Avg Rating', 'Num Restaurants']))

,Location,Avg Rating,Num Restaurants
0,Lavelle Road,4.19,442
1,Koramangala 5th Block,4.15,1782
2,Sankey Road,4.11,17
3,Koramangala 3rd Block,4.10,161
4,Cunningham Road,4.10,333
5,St. Marks Road,4.10,304
6,Koramangala 2nd Block,4.07,45
7,Sadashiv Nagar,4.06,38
8,Church Street,4.05,506
9,Residency Road,4.05,440


Over-saturated Locations

In [5]:
q2 = """
SELECT location, COUNT(*) AS restaurant_count, ROUND(AVG(rate), 2) AS avg_rating
FROM restaurants
GROUP BY location
HAVING COUNT(*) > (SELECT AVG(cnt)*1.5 FROM (SELECT COUNT(*) as cnt FROM restaurants GROUP BY location) t)
ORDER BY restaurant_count DESC
LIMIT 10;
"""
display(run_query(q2, ['Location', 'Restaurant Count', 'Avg Rating']))

,Location,Restaurant Count,Avg Rating
0,Koramangala 5th Block,1782,4.15
1,BTM,1454,3.76
2,Indiranagar,1346,3.96
3,HSR,1161,3.84
4,Jayanagar,1037,3.94
5,JP Nagar,1016,3.86
6,Whitefield,824,3.82
7,Koramangala 7th Block,723,3.99
8,Koramangala 6th Block,718,3.95
9,Marathahalli,680,3.73


Online Order Impact

In [6]:
q3 = """
SELECT 
    CASE WHEN online_order = 1 THEN 'With Online Order' ELSE 'Without' END AS feature,
    ROUND(AVG(rate),2) AS avg_rating, COUNT(*) AS num_restaurants
FROM restaurants GROUP BY online_order;
"""
display(run_query(q3, ['Feature', 'Avg Rating', 'Count']))

,Feature,Avg Rating,Count
0,Without,3.93,6823
1,With Online Order,3.89,16335


Table Booking Impact

In [7]:
q4 = """
SELECT 
    CASE WHEN book_table = 1 THEN 'With Table Booking' ELSE 'Without' END AS feature,
    ROUND(AVG(rate),2) AS avg_rating, COUNT(*) AS num_restaurants
FROM restaurants GROUP BY book_table;
"""
display(run_query(q4, ['Feature', 'Avg Rating', 'Count']))

,Feature,Avg Rating,Count
0,Without,3.81,17057
1,With Table Booking,4.16,6101


Best Price Segment

In [8]:
q5 = """
SELECT price_segment, ROUND(AVG(rate),2) AS avg_rating, COUNT(*) AS num_restaurants
FROM restaurants GROUP BY price_segment ORDER BY avg_rating DESC;
"""
display(run_query(q5, ['Price Segment', 'Avg Rating', 'Count']))

,Price Segment,Avg Rating,Count
0,Ultra-Premium,4.23,1554
1,Premium,4.09,5114
2,Low,3.82,6785
3,Mid,3.81,9705


Top Cuisines

In [9]:
q6 = """
SELECT TRIM(SUBSTRING_INDEX(SUBSTRING_INDEX(cuisines, ',', numbers.n), ',', -1)) AS cuisine,
       COUNT(*) AS num_restaurants
FROM restaurants
JOIN (SELECT 1 n UNION SELECT 2 UNION SELECT 3 UNION SELECT 4 UNION SELECT 5) numbers
GROUP BY cuisine
ORDER BY num_restaurants DESC
LIMIT 10;
"""
display(run_query(q6, ['Cuisine', 'Number of Restaurants']))

,Cuisine,Number of Restaurants
0,North Indian,17490
1,Chinese,13824
2,Fast Food,7698
3,Continental,6064
4,Desserts,5500
5,South Indian,5115
6,Beverages,4871
7,Cafe,4860
8,Biryani,4591
9,Italian,4287


Revenue by Location (using Orders)

In [10]:
q7 = """
SELECT r.location, ROUND(SUM(o.order_value),2) AS total_revenue, COUNT(o.order_id) AS total_orders
FROM orders o
JOIN restaurants r ON o.restaurant_name = r.restaurant_name
GROUP BY r.location
ORDER BY total_revenue DESC
LIMIT 10;
"""
display(run_query(q7, ['Location', 'Total Revenue', 'Total Orders']))

,Location,Total Revenue,Total Orders
0,Koramangala 5th Block,13874337.05,13862
1,BTM,11423063.80,11527
2,Indiranagar,10609319.14,10717
3,HSR,9489621.42,9519
4,Jayanagar,7925331.81,8109
5,JP Nagar,7845820.42,7919
6,Whitefield,6443756.59,6565
7,Koramangala 6th Block,5847029.60,5780
8,Koramangala 7th Block,5602128.28,5585
9,Koramangala 4th Block,5461979.48,5485


Top Restaurants by Revenue

In [11]:
q8 = """
SELECT restaurant_name, ROUND(SUM(order_value),2) AS total_revenue, COUNT(*) AS num_orders
FROM orders 
GROUP BY restaurant_name
ORDER BY total_revenue DESC
LIMIT 10;
"""
display(run_query(q8, ['Restaurant Name', 'Total Revenue', 'Number of Orders']))

,Restaurant Name,Total Revenue,Number of Orders
0,Kesar Sweet Shop and Fast Food,25222.36,20
1,Khan Saheb Grills and Rolls,23507.24,25
2,Bob's Bar,19518.14,15
3,Cake Cafe,19335.17,19
4,Biryani Mane,19249.45,15
5,Hungry Lee,19167.53,18
6,Andhra Grills,19153.72,19
7,Mighty Paws,18410.50,17
8,KKR Foodies,18137.18,19
9,Delhi Ke Bawarchi,18131.68,15


Payment Method Performance

In [12]:
q9 = """
SELECT payment_method, ROUND(AVG(order_value),2) AS avg_order_value, COUNT(*) AS num_orders
FROM orders GROUP BY payment_method;
"""
display(run_query(q9, ['Payment Method', 'Avg Order Value', 'Number of Orders']))

,Payment Method,Avg Order Value,Number of Orders
0,Card,989.02,8364
1,Cash,983.47,8384
2,UPI,985.75,8252


High Performing Restaurants (Rating + Revenue)

In [13]:
q10 = """
SELECT r.restaurant_name, r.location, ROUND(AVG(r.rate),2) AS avg_rating, 
       ROUND(SUM(o.order_value),2) AS total_revenue
FROM restaurants r
JOIN orders o ON r.restaurant_name = o.restaurant_name
GROUP BY r.restaurant_name, r.location
HAVING avg_rating >= 4.0
ORDER BY total_revenue DESC
LIMIT 10;
"""
display(run_query(q10, ['Restaurant', 'Location', 'Avg Rating', 'Total Revenue']))

,Restaurant,Location,Avg Rating,Total Revenue
0,Hammered,Cunningham Road,4.69,536073.92
1,Tiger Trail - Ramada Hotel,Shivajinagar,4.00,377707.38
2,Bonsouth,Koramangala 5th Block,4.20,262617.81
3,YORK St.,Koramangala 5th Block,4.40,261956.16
4,Roundup Cafe,Koramangala 5th Block,4.20,249621.54
5,Panchavati Gaurav Thali,Brigade Road,4.00,245374.43
6,The Beer Cafe,Koramangala 4th Block,4.20,237173.92
7,Infinitea Tea Room & Tea Store,Cunningham Road,4.27,236899.98
8,Buff Buffet Buff,Koramangala 5th Block,4.50,235269.02
9,Red Onion,Shanti Nagar,4.24,234780.71
